# NIFTY-50 Investment Intelligence Platform
## Complete Analysis Notebook

**Cult Open Projects 2026** — Data-Driven Investment Intelligence Using NIFTY-50 Market Data

This notebook documents the complete end-to-end analysis pipeline:
1. Data Loading & Cleaning
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. Prediction Engine (Ridge, XGBoost, LightGBM)
5. Portfolio Construction (Conservative / Balanced / Aggressive)
6. Risk Assessment
7. Explainable AI (SHAP)
8. Anomaly Detection
9. Results Summary & Key Insights

---

**Author:** Shiv Prakash Vishwari | IIT Roorkee


## 1. Setup & Data Loading

In [ ]:
import os, json, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.4f}".format)

PROJ_ROOT = ".."
DATA_RAW = os.path.join(PROJ_ROOT, "data", "raw")
DATA_PROC = os.path.join(PROJ_ROOT, "data", "processed")
OUTPUT_DIR = os.path.join(PROJ_ROOT, "outputs")
MODEL_DIR = os.path.join(PROJ_ROOT, "models")

print("Setup complete.")

### 1.1 Load Processed Features

The data processing pipeline (`src/data_processing.py`) performs:
- Loading 52 individual stock CSVs + metadata
- Cleaning: deduplication, forward-fill (limit=3), drop NaN Close
- Feature engineering: 56 columns across 6 categories
- Target creation: 21-day forward return and binary direction

In [ ]:
df = pd.read_parquet(os.path.join(DATA_PROC, "nifty50_features.parquet"))
print(f"Shape: {df.shape}")
print(f"Stocks: {df['Symbol'].nunique()}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Industries: {df['Industry'].nunique()}")
df.head()

In [ ]:
df.info(verbose=False)
print("\nColumn types:")
print(df.dtypes.value_counts())
print(f"\nMemory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

## 2. Exploratory Data Analysis

### 2.1 Dataset Overview

In [ ]:
summary = pd.DataFrame({
    "Metric": ["Total Records", "Unique Symbols", "Industries", "Date Range",
               "Feature Columns", "Missing %"],
    "Value": [f"{len(df):,}", df["Symbol"].nunique(), df["Industry"].nunique(),
              f"{df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}",
              df.shape[1], f"{df.isnull().mean().mean():.2%}"]
})
summary

### 2.2 Sector Distribution

In [ ]:
sector_counts = df.drop_duplicates("Symbol")["Industry"].value_counts()
fig = px.pie(values=sector_counts.values, names=sector_counts.index,
             title="NIFTY-50 Sector Distribution",
             color_discrete_sequence=px.colors.qualitative.Set3)
fig.update_layout(height=450)
fig.show()

### 2.3 Price Evolution — Top 10 Stocks

In [ ]:
recent = df[df["Date"] >= "2005-01-01"]
avg_mcap = recent.groupby("Symbol").apply(lambda g: (g["Close"] * g["Volume"]).mean())
top10 = avg_mcap.nlargest(10).index

fig = go.Figure()
for sym in top10:
    s = df[df["Symbol"] == sym].set_index("Date")["Close"]
    normed = s / s.iloc[0]
    fig.add_trace(go.Scatter(x=normed.index, y=normed.values, name=sym, mode="lines"))

fig.update_layout(title="Normalized Price Evolution — Top 10 Stocks",
                  yaxis_title="Normalized Price (base=1)", height=500)
fig.show()

### 2.4 Return Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (col, title) in enumerate([
    ("ret_1d", "Daily Returns"), ("ret_21d", "21-Day Returns"), ("ret_63d", "63-Day Returns")
]):
    data = df[col].dropna()
    axes[i].hist(data.clip(-0.3, 0.3), bins=100, density=True, alpha=0.7, color="steelblue")
    axes[i].set_title(title, fontsize=13)
    axes[i].axvline(0, color="red", linestyle="--", linewidth=0.8)
    axes[i].set_xlabel("Return")
    stats_text = f"Mean: {data.mean():.4f}\nStd: {data.std():.4f}\nSkew: {data.skew():.2f}\nKurt: {data.kurtosis():.1f}"
    axes[i].text(0.95, 0.95, stats_text, transform=axes[i].transAxes, va="top", ha="right",
                 fontsize=8, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
plt.tight_layout()
plt.show()

### 2.5 Market Volatility Over Time

In [ ]:
mkt_vol = df.groupby("Date")["vol_21d"].mean()
fig = go.Figure()
fig.add_trace(go.Scatter(x=mkt_vol.index, y=mkt_vol.values, fill="tozeroy",
                          fillcolor="rgba(220,50,50,0.1)", line=dict(color="crimson", width=1)))
fig.update_layout(title="Market-Wide 21-Day Annualized Volatility",
                  yaxis_title="Volatility", height=400)
fig.show()

### 2.6 Sector Performance Boxplot

In [ ]:
sector_rets = df[["Industry", "ret_21d"]].dropna()
order = sector_rets.groupby("Industry")["ret_21d"].median().sort_values(ascending=False).index

fig = px.box(sector_rets, x="Industry", y="ret_21d", category_orders={"Industry": list(order)},
             title="21-Day Return Distribution by Sector", color="Industry")
fig.update_layout(height=500, showlegend=False)
fig.show()

### 2.7 Feature Correlation Matrix

In [ ]:
feat_cols = ["ret_1d", "ret_5d", "ret_21d", "RSI_14", "MACD_hist", "BB_pctB",
             "vol_21d", "vol_ratio", "ATR_pct", "momentum_21", "drawdown"]
corr = df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
            ax=ax, square=True, linewidths=0.5)
ax.set_title("Feature Correlation Matrix", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Feature Engineering

56 features engineered from raw OHLCV data, organized into 6 categories:

| Category | Features | Count |
|----------|----------|-------|
| Returns | 1d, 5d, 10d, 21d, 63d log-returns | 5 |
| Moving Averages | SMA/EMA (5,10,20,50,200) + price-to-MA ratios | 13 |
| Oscillators | RSI-14, MACD (line/signal/hist), Bollinger %B | 5 |
| Volatility | 10d/21d/63d annualized vol, ATR%, volume ratio | 5 |
| Momentum | 10d/21d momentum, drawdown from peak | 3 |
| Calendar | Day of week, month | 2 |

**Target variables:**
- `fwd_ret_21d`: 21-day forward log-return (regression)
- `fwd_direction`: binary (1 if fwd_ret_21d > 0) (classification)

All features are computed per-stock with rolling windows — no future data leakage.

In [ ]:
# Show feature columns available
print("All columns:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col}")

## 4. Stock Prediction Engine

### 4.1 Data Split Strategy

We use a strict **time-based split** to prevent lookahead bias:
- **Train:** before 2018-01-01 (181,691 rows)
- **Validation:** 2018-01 to 2019-06 (18,032 rows)
- **Test:** 2019-07 to 2020-12 (18,375 rows)

The test period deliberately includes the COVID-19 market crash — a challenging but realistic evaluation scenario.

In [ ]:
FEATURE_COLS = [
    "ret_1d", "ret_5d", "ret_10d", "ret_21d", "ret_63d",
    "price_to_SMA20", "price_to_SMA50", "price_to_SMA200",
    "RSI_14", "MACD_line", "MACD_signal", "MACD_hist", "BB_pctB",
    "vol_10d", "vol_21d", "vol_63d", "vol_ratio", "ATR_pct",
    "momentum_10", "momentum_21", "drawdown", "dow", "month",
]

TARGET_REG = "fwd_ret_21d"
TARGET_CLF = "fwd_direction"

modeling_df = df.dropna(subset=FEATURE_COLS + [TARGET_REG, TARGET_CLF])

train = modeling_df[modeling_df["Date"] < "2018-01-01"]
val = modeling_df[(modeling_df["Date"] >= "2018-01-01") & (modeling_df["Date"] < "2019-07-01")]
test = modeling_df[(modeling_df["Date"] >= "2019-07-01") & (modeling_df["Date"] <= "2020-12-31")]

print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

X_train, y_train = train[FEATURE_COLS].values, train[TARGET_REG].values
X_val, y_val = val[FEATURE_COLS].values, val[TARGET_REG].values
X_test, y_test = test[FEATURE_COLS].values, test[TARGET_REG].values

### 4.2 Model Results

We load pre-trained model results (training performed in `src/prediction_engine.py`).

In [ ]:
with open(os.path.join(OUTPUT_DIR, "model_results.json")) as f:
    results = json.load(f)

# Regression results
reg_df = pd.DataFrame(results["regression"]).T
reg_df.columns = [c.replace("_", " ").title() for c in reg_df.columns]
print("=== Regression Models (21-Day Return Prediction) ===")
reg_df

In [ ]:
# Classification results
clf_df = pd.DataFrame(results["classification"]).T
clf_df.columns = [c.replace("_", " ").title() for c in clf_df.columns]
print("=== Classification Models (Direction Prediction) ===")
clf_df

In [ ]:
# Visual comparison
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=["Test RMSE (lower=better)", "Test R²", "Test Dir. Accuracy"])

models = list(results["regression"].keys())
colors = ["#636EFA", "#EF553B", "#00CC96"]

for i, (metric, _) in enumerate([("test_RMSE", "RMSE"), ("test_R2", "R²"), ("test_DirAcc", "Dir Acc")]):
    vals = [results["regression"][m][metric] for m in models]
    fig.add_trace(go.Bar(x=models, y=vals, marker_color=colors,
                          text=[f"{v:.4f}" for v in vals], textposition="outside"),
                  row=1, col=i+1)

fig.update_layout(height=400, showlegend=False, title="Model Comparison")
fig.show()

### 4.3 Key Findings

- **Ridge outperforms tree models on test data** — the primary predictive signal is linear (momentum + mean reversion)
- Tree models slightly overfit to non-linear patterns that don't generalize through the COVID regime shift
- **57.7% directional accuracy** suggests a modest but useful predictive signal. Stock-market return prediction is inherently noisy, and low R² values are expected in financial return forecasting. Directional accuracy is more meaningful for decision support than R² alone.
- The prediction engine is designed as one input into the broader intelligence platform, not as a standalone price predictor


## 5. Portfolio Construction

### 5.1 Methodology

**Markowitz Mean-Variance Optimization** maximizing:

$$U = \mu^T w - \frac{\lambda}{2} w^T \Sigma w$$

Subject to profile-specific constraints on volatility, concentration, and diversification.

In [ ]:
profiles = {}
for profile in ["Conservative", "Balanced", "Aggressive"]:
    path = os.path.join(OUTPUT_DIR, f"portfolio_{profile.lower()}.csv")
    if os.path.exists(path):
        profiles[profile] = pd.read_csv(path)

# Summary
summary_rows = []
for name, port in profiles.items():
    summary_rows.append({
        "Profile": name,
        "Stocks": len(port),
        "Avg Return": f"{port['Ann_Return'].mean():.2%}",
        "Avg Vol": f"{port['Ann_Vol'].mean():.2%}",
        "Avg Sharpe": f"{port['Sharpe'].mean():.2f}",
        "Avg MaxDD": f"{port['Max_DD'].mean():.2%}",
    })
pd.DataFrame(summary_rows)

In [ ]:
# Portfolio allocation visualization
fig = make_subplots(rows=1, cols=3, specs=[[{"type": "pie"}]*3],
                    subplot_titles=list(profiles.keys()))

for i, (name, port) in enumerate(profiles.items()):
    fig.add_trace(go.Pie(labels=port["Symbol"], values=port["Weight_pct"],
                          textinfo="label+percent", textfont_size=9),
                  row=1, col=i+1)

fig.update_layout(height=400, title="Portfolio Allocations by Profile", showlegend=False)
fig.show()

In [ ]:
# Sector allocation comparison
fig = go.Figure()
for name, port in profiles.items():
    sector = port.groupby("Industry")["Weight_pct"].sum().reset_index()
    fig.add_trace(go.Bar(x=sector["Industry"], y=sector["Weight_pct"], name=name))

fig.update_layout(barmode="group", title="Sector Allocation by Profile",
                  yaxis_title="Weight %", height=450)
fig.show()

## 6. Risk Assessment

12 risk metrics computed per stock over full trading history.

In [ ]:
risk_df = pd.read_csv(os.path.join(OUTPUT_DIR, "risk_report.csv"))
print(f"Stocks analyzed: {len(risk_df)}")

# Top 10 by Sharpe
cols = ["Name", "Ann_Return", "Ann_Volatility", "Sharpe_Ratio", "Sortino_Ratio",
        "Max_Drawdown", "VaR_95", "Beta"]
risk_df.nlargest(10, "Sharpe_Ratio")[cols]

In [ ]:
# Risk-return scatter
fig = px.scatter(risk_df, x="Ann_Volatility", y="Ann_Return", text="Name",
                 size="Sharpe_Ratio", color="Sharpe_Ratio",
                 color_continuous_scale="RdYlGn",
                 title="Risk-Return Profile of NIFTY-50 Stocks",
                 hover_data=["Max_Drawdown", "Beta"])
fig.update_traces(textposition="top center", textfont_size=7)
fig.update_layout(height=600)
fig.show()

In [ ]:
# VaR distribution
fig = px.histogram(risk_df, x="VaR_95", nbins=30, title="Daily VaR(95%) Distribution",
                   labels={"VaR_95": "Value at Risk (95%)"})
fig.show()

## 7. Explainable AI — SHAP Analysis

SHAP (SHapley Additive exPlanations) values computed via TreeExplainer on the LightGBM regression model provide both global and local interpretability.

In [ ]:
shap_df = pd.read_csv(os.path.join(OUTPUT_DIR, "shap_importance.csv"))

fig = px.bar(shap_df, x="mean_abs_shap", y="feature", orientation="h",
             title="SHAP Feature Importance — Top Drivers of 21-Day Return Prediction",
             labels={"mean_abs_shap": "Mean |SHAP Value|", "feature": "Feature"})
fig.update_layout(yaxis=dict(autorange="reversed"), height=500)
fig.show()

In [ ]:
# Feature importance comparison: LightGBM native vs SHAP
fi_df = pd.read_csv(os.path.join(OUTPUT_DIR, "feature_importance.csv"))

fig = make_subplots(rows=1, cols=2, subplot_titles=["LightGBM Gain Importance", "SHAP Importance"])

fig.add_trace(go.Bar(x=fi_df["importance"].head(10), y=fi_df["feature"].head(10),
                      orientation="h", marker_color="#636EFA"), row=1, col=1)
fig.add_trace(go.Bar(x=shap_df["mean_abs_shap"].head(10), y=shap_df["feature"].head(10),
                      orientation="h", marker_color="#EF553B"), row=1, col=2)

fig.update_yaxes(autorange="reversed")
fig.update_layout(height=450, showlegend=False, title="Feature Importance: Native vs SHAP")
fig.show()

### Key Explainability Findings

1. **Month** is the strongest global predictor — capturing Indian market seasonality (pre-budget rally, monsoon effects, FII flows)
2. **ATR%** indicates volatility regime is critical — high-volatility periods have different return dynamics
3. **63-day return** captures medium-term momentum — consistent with academic literature on 3-12 month momentum
4. **Price-to-SMA200** captures long-term trend positioning — stocks above their 200-day average tend to continue trending

## 8. Market Anomaly Detection

Four independent detectors identify unusual market conditions.

In [ ]:
anomalies = pd.read_csv(os.path.join(OUTPUT_DIR, "anomalies.csv"))
anomalies["Date"] = pd.to_datetime(anomalies["Date"])

summary = anomalies["anomaly_type"].value_counts()
fig = px.bar(x=summary.index, y=summary.values, color=summary.index,
             title="Anomaly Events by Type",
             labels={"x": "Anomaly Type", "y": "Count"})
fig.show()

In [ ]:
# Timeline of anomalies by year
anomalies["Year"] = anomalies["Date"].dt.year
yearly = anomalies.groupby(["Year", "anomaly_type"]).size().reset_index(name="Count")

fig = px.bar(yearly, x="Year", y="Count", color="anomaly_type", barmode="stack",
             title="Anomaly Events by Year and Type")
fig.update_layout(height=450)
fig.show()

### Anomaly Insights

- **2008 (GFC)** and **2020 (COVID-19)** show the highest anomaly density across all categories
- **Volatility spikes are leading indicators** — they precede crash events by 5-10 trading days
- Single-day crashes are concentrated in crisis periods but occur sporadically in normal markets too
- Volume surges often coincide with earnings announcements and index rebalancing events

## 9. Results Summary & Key Insights

### Performance Summary

| Module | Key Metric | Value |
|--------|-----------|-------|
| Prediction | Best Dir. Accuracy | 57.71% |
| Prediction | Best RMSE | 0.136 |
| Portfolio | Sharpe (all profiles) | >1.6 |
| Risk | Metrics per stock | 12 |
| Explainability | SHAP features | 23 |
| Anomaly | Detectors | 4 |
| Dashboard | Pages | 7 |

### Key Insights

1. **Linear models generalize better during regime shifts** — Ridge outperforms XGBoost/LightGBM when the COVID-19 period breaks historical non-linear patterns
2. **57.7% directional accuracy suggests a modest but useful predictive signal** — stock-market return prediction is inherently noisy and low R² values are expected; directional accuracy is more meaningful for decision support than R² alone
3. **Calendar seasonality is the strongest SHAP predictor** — month-of-year effects are well-documented in Indian equity markets
4. **Anomaly detection provides useful risk-management signals** — volatility spikes tend to precede crash events by several trading days
5. **Mean-variance optimization with profile constraints produces diversified portfolios** — all profiles achieve historical Sharpe > 1.6, though these are based on lookback statistics and not forward-looking guarantees

### Limitations

- Survivorship bias in the stock universe
- No transaction costs modeled
- Fixed 21-day horizon
- Static risk-free rate assumption
- Limited to provided dataset (no fundamental/news data)

---

*NIFTY-50 Investment Intelligence Platform — Cult Open Projects 2026*